In [ ]:
from common import *

In [ ]:
# most: nastavljamo tacno odatle gde je sekcija 5 zavrsila (fizicke greske uklonjene)
data = loadData("backups/weatherAusAfter5_2.csv")

## 6. Analiza skupa podataka

### 6.1 Osnovne informacije

In [ ]:
data.head()

Pomoću funkcije head() vidimo prvih 5 redova(kao da ih gledamo u Excelu) i već u prvih 5 redova u ogromnom broju kolona vidimo NA vrednosti(u nekim kolonama vidimo da su za prvih pet redova sve vrednosti NA). Kako su one vrlo problematične za naše predviđanje, moramo da odradimo njihovu detaljnu analizu i da odlučimo na koji ćemo dalje način i da li ćemo uopšte koristiti ove vrednosti u analizi(da li ćemo odbaciti čitave kolone ili ćemo ih popuniti na neki način).

In [ ]:
data.info(
    verbose=True,
    memory_usage=True,
    show_counts=True
)

Ovaj set podataka sadrži 145460 redova i 23 kolone, od kojih su 16 kolona realnog tipa i 7 kolona tipa object. Neke od ovih kolona tipa object je moguće prebaciti u kategorijske radi brzine i efikasnosti(npr. RainToday i RainTomorrow). Takođe, zanimljivo je što ovaj set podataka je veličine preko 25.5MB, što je u skladu sa brojem redova koji ima.

### 6.2 Nedostajuće vrednosti

In [ ]:
data.isnull().sum()

In [ ]:
kolone_sa_na = data.columns[data.isna().any()].tolist()
na_po_lokaciji_pct = data.groupby("Location")[kolone_sa_na].apply(lambda x: x.isna().mean() * 100).round(1)
na_po_lokaciji_pct.style.background_gradient(cmap="Reds", axis=None).format(precision=1)

Kolone Sunshine (48%), Evaporation (43%), Cloud9am (38%) i Cloud3pm (41%) imaju daleko najveći procenat nedostajućih vrednosti u čitavom skupu podataka, dok su kolone poput Pressure9am/3pm (~10%) i WindGustDir/Speed (~7%) znatno manje pogođene, a osnovne kolone (MinTemp, MaxTemp, Rainfall, RainToday...) imaju ispod 2-3% NA vrednosti.

Kada se ti nedostaci raspodele po lokacijama za sve kolone koje imaju NA vrednosti (tabela ispod, u procentima), vidimo da za Sunshine, 19 od 49 stanica ima preko 90% nedostajućih vrednosti, dok 13 stanica ima manje od 10% - vrednosti su ili gotovo potpuno prisutne, ili gotovo potpuno odsutne za datu lokaciju, a retko su u sredini. Isti obrazac važi i za Evaporation i obe Cloud kolone. Za razliku od toga, kolone kao što su MinTemp, MaxTemp, Pressure9am/3pm, WindGustSpeed imaju procenat NA koji je nizak i relativno ravnomeran po svim lokacijama, bez tog obrasca "sve ili ništa".

Ovakva raspodela ukazuje da uzrok nedostajućih podataka nije vremenska pojava (npr. da neki region ima manje kiše ili oblaka), već problem može predstavljati npr. razlika u opremi i tipu meteorološke stanice:


Zašto je ovo bitno za predviđanje? Ovo nisu nedostaci MCAR tipa, već zavise od lokacije. To znači da:

globalna imputacija (npr. srednjom vrednošću cele kolone) nije adekvatna, jer bi za stanice koje nikada nisu merile Sunshine unosila potpuno izmišljenu vrednost;
imputacija bi trebalo da se radi po lokaciji (ili grupi sličnih lokacija/klimatskih regiona), a za stanice sa 100% NA to i dalje nije moguće bez spoljašnjih podataka;
za kolone gde je čitava lokacija bez podataka, realnije opcije su: izbacivanje kolone iz modela, izbacivanje tih lokacija, ili uvođenje binarnog obeležja "stanica poseduje instrument" kao dodatne informacije modelu;
brisanje svih redova sa bar jednom NA vrednošću nije opcija jer bismo izgubili većinu skupa podataka, s obzirom da čak četiri kolone imaju 38-48% nedostajućih vrednosti.

### 6.3 Raspon vrednosti promenljivih

Kreirajmo boxplot grafik pomoću koga ćemo moći primetiti u kojem se rasponu kreću vrednosti za svaku promenljivu.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=data, orient='h')
plt.xticks(rotation=90)
plt.show()

Sa ovog boxplot-a vidimo da su sve vrednosti potpuno različite i teško uporedive na istoj skali, jer su izražene u različitim jedinicama: Pressure9am/Pressure3pm su u hPa i kreću se oko 980-1040, Humidity9am/Humidity3pm su u procentima (0-100%), Rainfall je u mm (0-371), Sunshine u satima (0-14.5), WindGustSpeed/WindSpeed9am/WindSpeed3pm u km/h, temperature u °C, a Cloud9am/Cloud3pm u oktasima (0-9). Iz tog razloga ovaj boxplot i nije nešto koristan - pri daljoj analizi je neophodno skalirati podatke.

### 6.4 Opis obeležja

Analizirajmo osobine i raspodelu svakog obelžja pojedinačno, analiza će se razlikovati u zavisnosti od toga da li je promenljiva numerička ili kategorijska.

#### 6.4.1 Numerička obeležja

##### 6.4.1.1 MinTemp

MinTemp predstavlja minimalnu (najvišu) temperaturu tokom svakog dana merenja, izmerenu u Celzijusovim stepenima.

In [ ]:
data['MinTemp'].describe()

S obzirom da su medijana i srednja vrednost približni možemo pretpostaviti da je raspodela normalna, ali bi to trebalo proveriti. Takođe, minimalne i maksimalne vrednosti ne mora da znači da su outlier-i(moguće je imati toliku minimalnu temperaturu, za maksimalnu temperaturu poznata je situacija u Adelaidu 29. januara 2009.)

Prema zvaničnim BOM rekordima, najniža ikada izmerena temperatura u Australiji je -23.0°C (Charlotte Pass, 1994), a najviša 50.7°C (Onslow, 2022). Naš minimum od -8.5°C je duboko unutar tog opsega - realna vrednost bez potrebe za dodatnom proverom.

Analizirajmo raspodelu vrednosti ove promenljive.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['MinTemp'], kde=True)
plt.title('Distribucija minimalne temperature')
plt.xlabel('Minimalna temperatura')
plt.ylabel('Frekvencija')
plt.show()

Raspodela jeste približno normalna, ali ovakav grafik dobijamo zato što gledamo temperaturu na osnovu svih lokacija.

##### 6.4.1.2 MaxTemp
MaxTemp predstavlja maksimalnu (najvišu) temperaturu tokom svakog dana merenja, izmerenu u Celzijusovim stepenima.

In [ ]:
data['MaxTemp'].describe()

Mean (23.22°C) i medijana (22.6°C) su blizu, uz blagu desnu asimetriju što je i očekivano, jer češće imamo velike talase vrućine nego što hladni dani mogu da povuku prosek.

Minimum od -4.8°C i maksimum od 48.1°C ne moraju da budu greške, ali potrebno je to proveriti. Prema BOM rekordima, opseg -23.0°C do 50.7°C je granica onoga što je ikada izmereno u Australiji - naša oba ekstrema (-4.8°C i 48.1°C) su unutar tog opsega, dakle realne vrednosti.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['MaxTemp'], kde=True)
plt.title('Distribucija maksimalne temperature')
plt.xlabel('Maksimalna temperatura')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.3 Rainfall

Rainfall predstavlja količinu palih padavina tokom dana, izmerenu u milimetrima (mm). Histogram i analizu desne asimetrije ove promenljive smo već uradili ranije u notebook-u, pa ovde dopunjujemo sa describe(). 

Pomoću domenskog znanja možemo pretpostaviti da je vrednost ove promenljive tokom većine vremena jednaka nuli (u svim klimama osim tropske kiša ne pada toliko često), u situacijama kada je vrednost ove promenljive različita od nule, možemo pretpostaviti da će vrednost biti drastično veća. 

Nacrtajmo grafik raspodele i proverimo naše pretpostavke.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=data, x="Rainfall", bins=50)
plt.title("Rainfall - sirove vrednosti")
plt.show()

Sa ovog histograma možemo da uočimo da u najvećem broju merenja je bilo bez kiše, a tek u nekim možda ekstremnim situacijama(npr. oluja ili neki ciklon) smo imali ekstremne vrednosti.
Imamo jaku desnu asimteriju, tako da bi možda bilo bolje ovu promenljivu tretirati kao to da li je kiša uopšte padala tog dana ili ne(što već imamo kao RainToday) ili možda logaritmovati kada je kiša padala, ali više o načinu tretiranja ove promenljive u nastavku.

In [ ]:
data['Rainfall'].describe()

Ovde je razlika između mean (2.36mm) i medijane (0mm) ogromna - čak i 75% percentil je samo 0.8mm. Ovo potvrđuje ono što smo ranije zaključili: većina dana nema kišu, a mali broj ekstremnih dana "vuče" prosek naviše. Maksimum od 371mm je izmeren u CoffsHarbour (obalna lokacija u NSW, poznata po "east coast low" sistemima - jakim olujama koje dolaze sa okeana), 7. novembra 2009. - fizički potpuno moguće, ne greška u merenju.

Zvanični australijski rekord za dnevnu količinu padavina je 907mm (Crohamhurst, Kvinslend, 1893). Naš maksimum od 371mm je dakle ozbiljna, ekstremna oluja, ali daleko od tog rekorda - realna vrednost.

Kao što smo radili i za prethodna obeležja, analizirajmo promene u raspodeli kada vrednosti posmatramo podeljene na lokacije, takođe pogledajmo šta će se desiti ako na podatke primenimo logaritamsku tranformaciju.

##### 6.4.1.4 Temp9am

Temp9am predstavlja temperaturu vazduha u 9 časova ujutru, izmerenu u Celzijusovim stepenima.

In [ ]:
data['Temp9am'].describe()

Mean (16.99°C) i medijana (16.7°C) su skoro identični - praktično simetrična raspodela, slično kao MinTemp.

Minimum od -7.2°C je izmeren u MountGinini, istog dana (13. jul 2016.) kada smo videli i minimum za MaxTemp - što je odličan znak unutrašnje konzistentnosti podataka (isti hladni talas se odražava na više povezanih kolona istovremeno). Maksimum od 40.2°C je u PearceRAAF (vojna baza kraj Pertha, poznata po vrelim letnjim danima), 12. januara 2014.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['Temp9am'], kde=True)
plt.title('Distribucija temperature u 9h')
plt.xlabel('Temp9am')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.5 Temp3pm

Temp3pm predstavlja temperaturu vazduha u 15 časova (3 popodne), izmerenu u Celzijusovim stepenima.

In [ ]:
data['Temp3pm'].describe()

Mean (21.68°C) i medijana (21.1°C) su blizu - blaga desna asimetrija, slično kao MaxTemp (što ima smisla, jer je Temp3pm skoro identičan MaxTemp, r = 0.99 iz korelacione matrice).

Minimum od -5.4°C je ponovo MountGinini, 13. jul 2016. - isti hladan dan se pojavljuje kao ekstrem u sve četiri temperaturske kolone, što je još jedna potvrda da su podaci međusobno konzistentni. Maksimum od 46.7°C je u Moree (unutrašnjost NSW, poznata po ekstremnim letnjim vrućinama), 12. februara 2017.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['Temp3pm'], kde=True)
plt.title('Distribucija temperature u 15h')
plt.xlabel('Temp3pm')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.6 Evaporation

Evaporation predstavlja količinu isparene vode tokom dana (merenu isparnim panjom), izraženu u milimetrima (mm).

In [ ]:
data['Evaporation'].describe()

Mean (5.47mm) je znatno veći od medijane (4.8mm), desna asimetrija, iako ne toliko ekstremna kao kod Rainfall.

Na osnovu zvaničnih podataka vezanih za Evaporation od strane Biroa za meteorologiju(BOM), najveća moguća vrednost je otprilike 25mm za Evaporation (nevezano za lokaciju). Vrednosti veće od 30mm bi trebalo uzeti sa posebnom rezervom jer su one najverovatnije greška.

In [ ]:
len(data[data['Evaporation'] > 30])

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['Evaporation'], kde=True)
plt.title('Distribucija isparavanja')
plt.xlabel('Evaporation')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.7 Sunshine

Sunshine predstavlja broj sati direktne sunčeve svetlosti tokom dana, izmeren heliografom.

In [ ]:
data['Sunshine'].describe()

Mean (7.61h) i medijana (8.4h) su blizu, skewness ≈ -0.5 - blaga leva asimetrija (malo više oblačnih/kratkih dana nego ekstremno sunčanih).

I minimum i maksimum su potpuno fizički logični: minimum od 0h (potpuno oblačan dan, Cobar, 7. januar 2009.) i maksimum od 14.5h (Mildura, 28. decembar 2015.) - 14.5h je blizu teorijskog maksimuma broja sati dnevne svetlosti u Australiji tokom leta (~14-15h), što znači da je taj dan bio potpuno vedar, bez oblaka, od izlaska do zalaska sunca.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['Sunshine'], kde=True)
plt.title('Distribucija sunčevih sati')
plt.xlabel('Sunshine')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.8 Numeričko obeležje WindGustSpeed

WindGustSpeed predstavlja brzinu najjačeg udara vetra tokom dana, izraženu u km/h.

In [ ]:
data['WindGustSpeed'].describe()

Mean (40.04 km/h) i medijana (39 km/h) su blizu, skewness ≈ 0.87 - umerena desna asimetrija (retki, ali jako izraženi udari vetra vuku prosek naviše).

Minimum od 6 km/h (Launceston, 11. jul 2013.) je sasvim miran dan bez vetra - fizički potpuno normalno. Maksimum od 135 km/h (NorahHead, 21. april 2015.) je ekstreman, ali plauzibilan - NorahHead je izložen obalski rt u NSW, poznat po jakim olujama i "east coast low" sistemima, a 135 km/h odgovara sili jake oluje/blage tropske ciklonske aktivnosti.

Zvanično najjači nesudarni (ne-tornado) udar vetra ikada izmeren u Australiji je 408 km/h (Ciklon Olivia, ostrvo Barrow, 1996). Naš maksimum od 135 km/h je duboko unutar realnog opsega - ozbiljna oluja, ali daleko od tog ekstrema.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['WindGustSpeed'], kde=True)
plt.title('Distribucija brzine udara vetra')
plt.xlabel('WindGustSpeed')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.9 WindSpeed9am

WindSpeed9am predstavlja brzinu vetra u 9 časova ujutru, izraženu u km/h.

In [ ]:
data['WindSpeed9am'].describe()

Mean (14.04 km/h) i medijana (13 km/h) su blizu, skewness ≈ 0.78 - umerena desna asimetrija, slično kao WindGustSpeed (očekivano, jer su ove dve kolone povezane).

Minimum od 0 km/h (Albury, 27. decembar 2008.) je potpuno miran jutarnji vazduh - normalno. Maksimum od 130 km/h (Newcastle, 18. januar 2017.) je izuzetno visok za trajnu (ne-udarnu) brzinu vetra. Taj dan je MaxTemp u Newcastle-u bio čak 41°C - kombinacija ekstremne vrućine i jakog vetra je poznata i opasna "fire weather" situacija u Australiji, što dodatno potkrepljuje da je ovo bio realan ekstreman dan, a ne greška u merenju (mada WindGustSpeed za taj dan nažalost nedostaje, pa nije moguće direktno unakrsno proveriti).

S obzirom da je zvanični rekord za udar vetra u Australiji 408 km/h (Ciklon Olivia, 1996), a 130 km/h je trajna (ne udarna) brzina, ovo je i dalje realna, iako ekstremna vrednost.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['WindSpeed9am'], kde=True)
plt.title('Distribucija brzine vetra u 9h')
plt.xlabel('WindSpeed9am')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.10 WindSpeed3pm

WindSpeed3pm predstavlja brzinu vetra u 15 časova, izraženu u km/h.

In [ ]:
data['WindSpeed3pm'].describe()

Mean (18.66 km/h) i medijana (19 km/h) su skoro identični, skewness ≈ 0.63 - blaga desna asimetrija, nešto blaža nego kod WindSpeed9am, što ima smisla jer popodnevni vetar obično duva ravnomernije (manje ekstremnih tišina) nego jutarnji.

Minimum od 0 km/h (Albury, 14. februar 2009.) je opet miran dan. Maksimum od 87 km/h (GoldCoast, 20. maj 2009.) je znatno umereniji ekstrem u odnosu na WindGustSpeed (135) i WindSpeed9am (130) - logično, jer je "gust" po definiciji trenutni udar i uvek veći od prosečne/trajne brzine vetra u bilo kom terminu merenja, i sasvim je unutar realnog opsega (australijski rekord za udar vetra je 408 km/h).

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['WindSpeed3pm'], kde=True)
plt.title('Distribucija brzine vetra u 15h')
plt.xlabel('WindSpeed3pm')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.11 Humidity9am

Humidity9am predstavlja relativnu vlažnost vazduha u 9 časova ujutru, izraženu u procentima (%).

In [ ]:
data['Humidity9am'].describe()

Mean (68.88%) i medijana (70%) su blizu, skewness ≈ -0.48 - blaga leva asimetrija (vlažnost je ograničena odozgo sa 100%, pa "rep" ka niskim vrednostima može da bude duži).

Minimum od 0% (Woomera, 20. oktobar 2013.) je ekstremno suv vazduh, ali plauzibilno za pustinjsku lokaciju. Maksimum od 100% (Albury, 31. jul 2010., sred zime) predstavlja potpuno zasićen vazduh (magla ili kiša) - takođe potpuno normalno. Relativna vlažnost je po definiciji ograničena na opseg 0-100%, pa ovde nema potrebe za poređenjem sa spoljašnjim rekordima - bilo koja vrednost unutar tog opsega je validna.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['Humidity9am'], kde=True)
plt.title('Distribucija vlažnosti u 9h')
plt.xlabel('Humidity9am')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.12 Humidity3pm

Humidity3pm predstavlja relativnu vlažnost vazduha u 15 časova, izraženu u procentima (%).

In [ ]:
data['Humidity3pm'].describe()

Mean (51.54%) i medijana (52%) su skoro identični, skewness ≈ 0.03 - praktično simetrična raspodela.

Odgovor na pitanje sa kraja prošlog dela: da, Humidity9am (mean 68.88%) je u proseku znatno veći od Humidity3pm (mean 51.54%) - razlika od skoro 17 procentnih poena. Ovo je meteorološki potpuno očekivano: ujutru je vazduh hladniji i bliži tački rošenja (rosne tačke), pa je relativna vlažnost veća, dok popodne, kad temperatura poraste, ista količina vodene pare u vazduhu odgovara nižoj relativnoj vlažnosti.

Minimum od 0% je opet u Woomera, istog dana (20. oktobar 2013.) kada je izmeren i minimum za Humidity9am - odličan znak konzistentnosti: taj dan je bio ekstremno suv i ujutru i popodne. Maksimum od 100% je u Albury, 26. februar 2012. Kao i kod Humidity9am, opseg 0-100% je definicioni, pa nema potrebe za poređenjem sa spoljašnjim rekordima.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['Humidity3pm'], kde=True)
plt.title('Distribucija vlažnosti u 15h')
plt.xlabel('Humidity3pm')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.13 Pressure9am

Pressure9am predstavlja atmosferski pritisak u 9 časova ujutru, sveden na nivo mora, izražen u hPa.

In [ ]:
data['Pressure9am'].describe()

Mean (1017.65 hPa) i medijana (1017.6 hPa) su praktično identični, skewness ≈ -0.1 - gotovo savršeno simetrična raspodela, što je uobičajeno za atmosferski pritisak.

Minimum od 980.5 hPa (NorfolkIsland, 11. jul 2009.) odgovara veoma dubokom ciklonu/oluji niskog pritiska - fizički potpuno moguće, čak i za relativno "blag" ekstrem (duboki cikloni idu i znatno ispod 950 hPa - zvanični rekord za najniži pritisak u australijskom regionu je oko 900 hPa, Ciklon Inigo 2003). Maksimum od 1041 hPa (Witchcliffe, 9. septembar 2011.) odgovara jakom anticiklonu (visokom pritisku), tipičnom za mirno, vedro vreme - takođe potpuno realno (tipičan gornji opseg u Australiji je do ~1040-1050 hPa).

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['Pressure9am'], kde=True)
plt.title('Distribucija pritiska u 9h')
plt.xlabel('Pressure9am')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.14 Pressure3pm

Pressure3pm predstavlja atmosferski pritisak u 15 časova, sveden na nivo mora, izražen u hPa.

In [ ]:
data['Pressure3pm'].describe()

Mean (1015.26 hPa) i medijana (1015.2 hPa) su praktično identični, skewness ≈ -0.05 - opet gotovo savršeno simetrično, u skladu sa Pressure9am.

Minimum od 977.1 hPa (Hobart, 12. jul 2016.) je izuzetno nizak pritisak - Tasmanija je krajem juna/početkom jula 2016. pogođena ozbiljnim nevremenom i poplavama, pa je ovo u skladu sa poznatim vremenskim događajem, ne greškom, i i dalje daleko od rekordnih ~900 hPa (ciklonski pritisak). Maksimum od 1039.6 hPa (Launceston, 22. jun 2010.) je jak zimski anticiklon.

Prosek Pressure3pm (1015.26) je nešto niži od Pressure9am (1017.65) - blaga, ali dosledna razlika, u skladu sa tim da pritisak ima dnevni ciklus (obično malo opada tokom popodneva usled zagrevanja vazduha).

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['Pressure3pm'], kde=True)
plt.title('Distribucija pritiska u 15h')
plt.xlabel('Pressure3pm')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.15 Cloud9am

Cloud9am predstavlja oblačnost neba u 9 časova ujutru, izraženu u oktasima (skala 0-9, gde 0 znači vedro nebo, a 9 potpuno naoblačeno).

In [ ]:
data['Cloud9am'].describe()

Ovo je (zajedno sa Cloud3pm) jedina numerička kolona koja je zapravo diskretna ordinalna skala (celi brojevi 0-9), a ne kontinualna veličina, pa describe() treba tumačiti s tom napomenom - mean (4.45) i skewness ovde nisu toliko informativni kao kod pravih kontinualnih veličina.

Minimum od 0 (potpuno vedro nebo, Albury, 16. decembar 2008.) i maksimum od 9 (potpuno oblačno, Sydney, 23. septembar 2009.) su prosto krajnje vrednosti skale, ne outlieri - očekivano je da se pojavljuju često, jer je nebo ili sasvim vedro ili sasvim oblačno prilično čest slučaj. Oktas skala je po definiciji ograničena na 0-9, pa ovde takođe nema potrebe za poređenjem sa spoljašnjim rekordima - jedino bi vrednost van tog opsega (npr. 10 ili -1) bila znak greške.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['Cloud9am'], bins=10, discrete=True)
plt.title('Distribucija oblačnosti u 9h')
plt.xlabel('Cloud9am (oktasi)')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.16 Cloud3pm

Cloud3pm predstavlja oblačnost neba u 15 časova, izraženu u oktasima (skala 0-9).

In [ ]:
data['Cloud3pm'].describe()

Mean (4.51) i medijana (5) su vrlo blizu vrednostima kod Cloud9am (4.45 i 5) - u proseku slična oblačnost ujutru i popodne, za razliku od temperature ili vlažnosti gde je razlika 9h/15h bila izraženija.

Minimum od 0 (Cobar, 16. januar 2009.) i maksimum od 9 (Woomera, 2. novembar 2012.) su opet samo krajnje vrednosti skale (0-9), ne greške.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['Cloud3pm'], bins=10, discrete=True)
plt.title('Distribucija oblačnosti u 15h')
plt.xlabel('Cloud3pm (oktasi)')
plt.ylabel('Frekvencija')
plt.show()

##### 6.4.1.17 Numerička obeležja u odnosu na RainTomorrow

Do sada smo svaku numeričku promenljivu posmatrali samo pojedinačno, bez poređenja sa ciljnom promenljivom. Pogledajmo sada kako se raspodela svake numeričke promenljive razlikuje u zavisnosti od toga da li sledećeg dana pada kiša.

In [ ]:
numericke_promenljive = ['MinTemp', 'MaxTemp', 'Rainfall', 'Temp9am', 'Temp3pm', 'Evaporation',
                          'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm',
                          'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm',
                          'Cloud9am', 'Cloud3pm']

podaci_sa_ciljem = data.dropna(subset=['RainTomorrow'])

cols = 4
rows = -(-len(numericke_promenljive) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
axes = axes.flatten()

for ax, kolona in zip(axes, numericke_promenljive):
    sns.boxplot(data=podaci_sa_ciljem, x='RainTomorrow', y=kolona, hue='RainTomorrow',
                palette='Set1', legend=False, ax=ax)
    ax.set_title(kolona, fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

for ax in axes[len(numericke_promenljive):]:
    ax.axis('off')

fig.suptitle('Numerička obeležja u odnosu na RainTomorrow', fontsize=14)
plt.tight_layout()
plt.show()

Najupadljiviji obrazac vidimo kod promenljivih vezanih za vlažnost vazduha i oblačnost. Humidity3pm pokazuje najjasnije razdvajanje od svih numeričkih promenljivih. Kada sutra neće padati kiša tipična vrednost je oko 47 procenata, a kada hoće tipična vrednost skače na oko 70 procenata, pa se kutije na grafiku jedva dodiruju. Humidity9am prati isti obrazac, samo nešto slabije izražen. Sunshine pokazuje skoro ogledalsku sliku ovoga, dani posle kojih sledi kiša imaju primetno manje sunčanih sati u proseku. Cloud9am i Cloud3pm prate istu logiku, veća naoblačenost tog dana prirodno prethodi kiši sledećeg dana. Pressure9am i Pressure3pm su takođe primetno niži pred kišne dane, što se poklapa sa osnovnim meteorološkim znanjem, sistemi niskog pritiska donose kišu. MaxTemp i Temp3pm pokazuju umereno, ali jasno vidljivo razdvajanje, dok WindGustSpeed i obe kolone brzine vetra pokazuju blaže, ali i dalje primetno više vrednosti pred kišne dane. Rainfall sama po sebi već pokazuje veće vrednosti pred kišno sutra, mada je to teško videti golim okom na ovom grafiku jer nekoliko ekstremnih dana potpuno razvlači skalu, obe grupe uglavnom imaju vrednosti blizu nule.

MinTemp i Temp9am jedva pokazuju ikakvu vidljivu razliku između dve grupe. Evaporation takođe pokazuje samo blagu razliku. Gledajući samo ove grafike moglo bi se zaključiti da jutarnja temperatura i minimalna dnevna temperatura nose skoro nikakvu informaciju o tome da li će sutra padati kiša.

Taj zaključak bio bi pogrešan ili bar nepotpun, i tu pomaže pogled koji uključuje još jednu promenljivu. Kada MinTemp podelimo ne samo po RainTomorrow nego i po mesecu, pojavljuje se sasvim drugačija slika. U svakom pojedinačnom mesecu medijana MinTemp na dan posle kog pada kiša je viša nego na dan koji ostaje suv, a ta razlika ume da bude tri stepena ili više u letnjim mesecima kao što su decembar, januar i februar, dok pada na manje od jednog stepena u septembru i oktobru. Razlog zašto ovaj prvi grafik to sakriva je taj što se meseci međusobno ogromno razlikuju po tipičnoj temperaturi, pa topao zimski dan i hladan letnji dan mogu da završe sa sličnom sirovom vrednošću MinTemp iako za rizik od kiše znače potpuno suprotne stvari. Kada se sezona uzme u obzir, ista promenljiva ponovo postaje informativna.

In [ ]:
mesecna_analiza = data.dropna(subset=['RainTomorrow']).copy()
mesecna_analiza['Month'] = pd.to_datetime(mesecna_analiza['Date']).dt.month

medijane_po_mesecu = mesecna_analiza.groupby(['Month', 'RainTomorrow'])['MinTemp'].median().unstack()
medijane_po_mesecu['Razlika'] = medijane_po_mesecu['Yes'] - medijane_po_mesecu['No']
medijane_po_mesecu.round(2)

Opšta pouka iz ovog dela je da promenljiva koja pokazuje slabo razdvajanje sama za sebe ne znači automatski da je beskorisna za model. Možda samo treba da se posmatra zajedno sa nekom drugom promenljivom, ovde mesec odnosno sezona, da bi njena veza sa kišom postala vidljiva. Ovo se uklapa i sa time što nekoliko izvedenih atributa opisanih ranije u ovom dokumentu, DayOfYear_Sin i DayOfYear_Cos, modelu već daju pristup sezonskoj informaciji. To verovatno pomaže da se iz temperaturskih kolona izvuče baš ovakav skriveni signal, iako ga sam grafik dve promenljive ne može direktno da pokaže.

#### 6.4.2 Kategorijska obeležja

In [ ]:
data.select_dtypes(include=['object', 'category'])

##### 6.4.2.1 Date

Kategorijsko obeležje Date predstavlja datum zabeleženog merenja različitih faktora uticajnosti na kišu na određenoj lokaciji. Više o samim jedinstvenim vrednostima ovog obeležja, njegove uticajnosti na našu ciljanu promenljivu RainTomorrow ćemo obraditi u nastavku.

In [ ]:
data['Date'].value_counts()

In [ ]:
counts = data['Date'].value_counts()
maxDates = counts[counts == 49].index
result = data[data['Date'].isin(maxDates)]
print(f'Najraniji datum kada je pocelo merenje na nekoj lokaciji: ', data['Date'].min())
print(f'Najraniji datum od kada imamo podatke na svim lokacijama: ', result['Date'].min())
print(f'Najkasniji datum kada se zavrsilo merenje na nekoj lokaciji: ', result['Date'].max())
print(f'Najkasniji datum kada se zavrsilo merenje na svim lokacijama: ', data['Date'].max())



Ono što je zanimljivo jeste da za različite datume imamo različite jedinstvene vrednosti. To je rezultat toga što nam od lokacije do lokacije zavisi kada je počelo merenje. Možemo primetiti da prve rezultate imamo od 1.11.2007. a da tek od 1.3.2013. imamo svakodnevne podatke za sve naše lokacije. Kraj se relativno podudara, razlika je samo jedan dan. Sam po sebi datum i nema nešto preterano mnogo značajnosti, međutim, može da nam bude dobra podloga za pretvaranje u godinu i mesec npr. gde bismo znali koje je godišnje doba u tom periodu i šta bismo mogli da očekujemo. Takva promenljiva bi više uticala na naš model nego sam datum.
Takođe, valjalo bi proveriti da li postoji neki dani gde bismo imali preskakanja(nemamo podatke za određeni dan) za određenu lokaciju, ako je merenje već počelo.

In [ ]:
def pronadji_nedostajuce_dane(df, location_col="Location", date_col="Date"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col]).dt.normalize()
    df = df.drop_duplicates(subset=[location_col, date_col])

    records = []
    for location, group in df.groupby(location_col, sort=True):
        dates = group[date_col].sort_values()
        start, end = dates.min(), dates.max()

        full_range = pd.date_range(start, end, freq="D")
        present = pd.Index(dates.unique())
        missing = full_range.difference(present)

        for missing_date in missing:
            records.append({
                "Location": location,
                "MissingDate": missing_date,
                "PeriodStart": start,
                "PeriodEnd": end,
            })

    result = pd.DataFrame(records)
    if not result.empty:
        result = result.sort_values(["Location", "MissingDate"]).reset_index(drop=True)

    return result



nedostajuci_dani = pronadji_nedostajuce_dane(data)
nedostajuci_dani

Vidimo da imamo 4112 redova sa nedostajućim vrednostima za neki dan otkako je započelo njihovo merenje. Sada ćemo napraviti kratak rezime po brojkama za koju lokaciju koliko dana fali.

In [ ]:
def rezime_nedostajucih_dana(missing_df):
    if missing_df.empty:
        return pd.DataFrame(columns=["Location", "PeriodStart", "PeriodEnd", "BrojFalecihDana", "FaleciDani"])

    summary = (
        missing_df
        .groupby("Location", as_index=False)
        .agg(
            PeriodStart=("PeriodStart", "first"),
            PeriodEnd=("PeriodEnd", "first"),
            BrojFalecihDana=("MissingDate", "count"),
            FaleciDani=("MissingDate", lambda x: sorted(x.dt.strftime("%Y-%m-%d").tolist())),
        )
        .sort_values("BrojFalecihDana", ascending=False)
    )
    return summary

rezime = rezime_nedostajucih_dana(nedostajuci_dani)
rezime

##### 6.4.2.2 Location

Kategorijsko obeležje Location predstavlja centralno obeležje u našem skupu podataka od kojih nam zavisi tretiranje svih ostalih promenljivih. Za svaku lokaciju imamo merenje za po jedan dan, a u zavisnosti od same lokacije i opremljenosti njene stanice, imamo dosta veliki broj nedostajućih vrednosti.

In [ ]:
data['Location'].value_counts()

Primećujemo da imamo od 1578 observacija do 3436, u zavisnosti od lokacije. Sada ćemo to predstaviti grafički.

In [ ]:
plt.figure(figsize=(10, 12))

order = data['Location'].value_counts().index

ax = sns.countplot(
    y='Location',
    data=data,
    order=order,
    palette='Set2',
    hue='Location',
    legend=False
)

plt.title('Distribucija lokacija')
plt.xlabel('Broj zapisa')
plt.ylabel('Lokacija')

plt.tight_layout()
plt.show()

Na osnovu ovog grafika zaključujemo da se radi o nominalnom kategorijskom obeležju. Sada je neophodno proveriti kako ove lokacije utiču na našu ciljnu promenljivu RainTomorrow.

In [ ]:
pd.crosstab(
    index=data['Location'],
    columns=data['RainTomorrow'],
    normalize='index'
) * 100

In [ ]:
ct = pd.crosstab(
    data['Location'],
    data['RainTomorrow'],
    normalize='index'
) * 100

ct.plot(
    kind='bar',
    stacked=True,
    figsize=(14,8),
    colormap='Set1'
)

plt.ylabel('Procenat (%)')
plt.xlabel('Lokacija')
plt.title('Procenat kiše sutra po lokaciji')
plt.xticks(rotation=90)
plt.legend(title='RainTomorrow')
plt.tight_layout()
plt.show()

Sa slike i sa grafika vidimo da imamo neke grupe jako sličnih procenata kada je kiša padala i kada nije. To nam može pomoći u kasnijoj analizi da možda nekako možemo da ih grupišemo(napravimo klastere).

##### 6.4.2.3 WindGustDir

WindGustDir predstavlja smer najjačeg udara vetra u 24 sata do ponoći.

In [ ]:
data['WindGustDir'] = data['WindGustDir'].astype('category')

In [ ]:
data['WindGustDir'].value_counts()

In [ ]:
plt.figure(figsize=(10, 12))

ax = sns.countplot(
    y='WindGustDir',
    data=data,
    palette='Set1',
    hue='WindGustDir',
    legend=False
)

plt.title('Distribucija udara vetrova')
plt.xlabel('Broj zapisa')
plt.ylabel('Udar vetra')

plt.tight_layout()
plt.show()

Na osnovu ovoga, vidimo da WindGustDir predstavlja nominalnu kategorijsku promenljivu. Najčešći pravac najjačih udara vetra je zapad (W). Ovakav rezultat je očekivan, jer se veliki deo Australije nalazi pod uticajem preovlađujućih zapadnih vetrova (westerlies) i hladnih frontova koji dolaze sa Južnog okeana. Ovi sistemi često izazivaju najjače udare vetra upravo iz zapadnog pravca, naročito u južnim i jugoistočnim delovima zemlje.

In [ ]:
ct = pd.crosstab(
    data['WindGustDir'],
    data['RainTomorrow'],
    normalize='index'
) * 100

ct.plot(
    kind='bar',
    stacked=True,
    figsize=(14,8),
    colormap='Set1'
)

plt.ylabel('Procenat (%)')
plt.xlabel('Pravac najjaceg udara vetra')
plt.title('Procenat kiše sutra po lokaciji')
plt.xticks(rotation=90)
plt.legend(title='RainTomorrow')
plt.tight_layout()
plt.show()

Sa ovog grafika ne možemo mnogo toga zaključiti sem da je odgovor RainTomorrow pretežno No u odnosu na WindGustDir u svim pravcima. Međutim, možda bismo nešto jasnije mogli da vidimo šta se dešava ukoliko bismo umesto 16 pravaca imali 8 glavnih(N, S, W, E, NE, NW, SE, SW).

In [ ]:
direction_map = {
    'N': 'N',
    'NNE': 'N',
    'NNW': 'N',

    'NE': 'NE',

    'E': 'E',
    'ENE': 'E',
    'ESE': 'E',

    'SE': 'SE',
    'SSE': 'S',

    'S': 'S',
    'SSW': 'S',

    'SW': 'SW',

    'W': 'W',
    'WNW': 'W',
    'WSW': 'W',

    'NW': 'NW'
}

data['WindGustDir_Group'] = data['WindGustDir'].map(direction_map)

ct = pd.crosstab(
    data['WindGustDir_Group'],
    data['RainTomorrow'],
    normalize='index'
) * 100

order = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
ct = ct.reindex(order)

ax = ct.plot(
    kind='bar',
    stacked=True,
    figsize=(10,6),
    color=['red', 'gray']
)

plt.title('RainTomorrow u odnosu na grupisani pravac najjačeg udara vetra')
plt.xlabel('Pravac vetra')
plt.ylabel('Procenat (%)')
plt.legend(title='RainTomorrow')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

I dalje nemamo neka prevelika odstupanja, s obzirom na to da vidimo da vrednosti na graficima idu od nekih ~75% do ~85%, gde se blago izdvaja E.

##### 6.4.2.4 WindDir9am

WindDir9am predstavlja smer vetra određen u 9 časova ujutru.

In [ ]:
data['WindDir9am'].value_counts()

Na ovakav raspored bi mogli uticati lokacije koje su obuhvaćene ovim skupom podataka(u kom delu zemlje se nalaze), kao i samo godišnje doba u kome su merenja odrađena.

In [ ]:
plt.figure(figsize=(10, 12))

ax = sns.countplot(
    y='WindDir9am',
    data=data,
    palette='Set1',
    hue='WindDir9am',
    legend=False
)

plt.title('Distribucija udara vetrova u 9 ujutru')
plt.xlabel('Broj zapisa')
plt.ylabel('Smer vetra u 9 ujutru')

plt.tight_layout()
plt.show()

Iz ovoga možemo zaključiti da imamo nominalnu kategorijsku promenljivu. Sada ćemo proveriti kako ona utiče na našu ciljnu promenljivu.

In [ ]:
ct = pd.crosstab(
    data['WindDir9am'],
    data['RainTomorrow'],
    normalize='index'
) * 100

ct.plot(
    kind='bar',
    stacked=True,
    figsize=(14,8),
    colormap='Set1'
)

plt.ylabel('Procenat (%)')
plt.xlabel('Pravac vetra u 9 ujutru')
plt.title('Procenat kiše sutra po lokaciji')
plt.xticks(rotation=90)
plt.legend(title='RainTomorrow')
plt.tight_layout()
plt.show()

Opet vidimo da i ovde ne možemo nešto mnogo zaključaka da donesemo, sem da na svim pravcima vetra u 9 ujutru pretežno nije bilo više kiše nego što jeste. Trebalo bi ispitati šta još dodatno utiče na našu ciljnu promenljivu pored pravca vetra u 9 ujutru da bismo mogli doneti više zaključaka.

##### 6.4.2.5 Kategorijsko obeležje WindDir3pm

WindDir3pm predstavlja smer vetra određen u 15 časova (3 popodne).

In [ ]:
data['WindDir3pm'].value_counts()

Najčešći smer vetra u 15h je jugoistok (SE), a najređi sever-severoistok (NNE). Na ovakav raspored, kao i kod prethodna dva obeležja pravca vetra, utiču lokacije obuhvaćene skupom podataka i godišnje doba merenja.

In [ ]:
plt.figure(figsize=(10, 12))

ax = sns.countplot(
    y='WindDir3pm',
    data=data,
    palette='Set1',
    hue='WindDir3pm',
    legend=False
)

plt.title('Distribucija pravca vetra u 15h')
plt.xlabel('Broj zapisa')
plt.ylabel('Smer vetra u 15h')

plt.tight_layout()
plt.show()

Iz ovoga možemo zaključiti da imamo nominalnu kategorijsku promenljivu. Sada ćemo proveriti kako ona utiče na našu ciljnu promenljivu.

In [ ]:
ct = pd.crosstab(
    data['WindDir3pm'],
    data['RainTomorrow'],
    normalize='index'
) * 100

ct.plot(
    kind='bar',
    stacked=True,
    figsize=(14,8),
    colormap='Set1'
)

plt.ylabel('Procenat (%)')
plt.xlabel('Pravac vetra u 15h')
plt.title('Procenat kiše sutra po pravcu vetra u 15h')
plt.xticks(rotation=90)
plt.legend(title='RainTomorrow')
plt.tight_layout()
plt.show()

Za razliku od WindGustDir i WindDir9am, ovde se ipak vidi jasniji obrazac: severni/severozapadni pravci popodnevnog vetra imaju oko 28% šanse za kišu sutradan, dok istočni pravci imaju oko 17%(skoro duplo manje). Ovo ima smisla jer u južnoj Australiji, severni/severozapadni vetar popodne često najavljuje dolazak hladnog fronta sa zapada/jugozapada, što je klasičan predznak kiše u narednim danima. WindDir3pm je zato verovatno korisnije obeležje za predviđanje od druga dva pomenuta.

##### 6.4.2.6 RainToday

RainToday predstavlja da li je na dan merenja bilo kiše (vrednost "Yes" ako je tog dana palo bar 1mm padavina, inače "No") - u suštini binarna verzija kolone Rainfall.

In [ ]:
data['RainToday'].value_counts()

Klase su neuravnotežene. Oko 77% dana je bez kiše ("No", 110319), a oko 22% sa kišom ("Yes", 31880) - u skladu sa onim što smo već utvrdili kod Rainfall (64%+ dana sa tačno 0mm).

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    x='RainToday',
    data=data,
    palette='Set1',
    hue='RainToday',
    legend=False
)

plt.title('Distribucija RainToday')
plt.xlabel('Da li je danas padala kiša')
plt.ylabel('Broj zapisa')

plt.tight_layout()
plt.show()

Ovo je binarna kategorijska promenljiva. Sada ćemo proveriti kako utiče na ciljnu promenljivu.

In [ ]:
ct = pd.crosstab(
    data['RainToday'],
    data['RainTomorrow'],
    normalize='index'
) * 100

ct.plot(
    kind='bar',
    stacked=True,
    figsize=(6,5),
    colormap='Set1'
)

plt.ylabel('Procenat (%)')
plt.xlabel('Da li je danas padala kiša')
plt.title('Procenat kiše sutra u odnosu na kišu danas')
plt.xticks(rotation=0)
plt.legend(title='RainTomorrow')
plt.tight_layout()
plt.show()

Za razliku od svih dosadašnjih kategorijskih obeležja gde je razlika u procentu kiše sutra između kategorija bila relativno mala, ovde je razlika ogromna. Ako danas nije padala kiša, šansa da sutra padne je samo 15.2%, a ako je danas padala kiša, ta šansa skače na 46.4% (skoro tri puta veća).

##### 6.4.2.7 RainTomorrow (ciljna promenljiva)

RainTomorrow je ciljna promenljiva koju predviđamo - da li će sledećeg dana padati kiša. Za razliku od ostalih kategorijskih obeležja, ovde nas prvenstveno zanima balans klasa, jer on direktno utiče na izbor metrika i strategiju treniranja modela (odeljak 12).

In [ ]:
data['RainTomorrow'].value_counts()

Klase su znatno neuravnotežene: oko 76% dana je bez kiše sutradan ("No") naspram oko 22% sa kišom ("Yes"). Odnos je identičan skoro kao kod RainToday (razlika je samo pomeranje serije za jedan dan).

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    x='RainTomorrow',
    data=data,
    palette='Set1',
    hue='RainTomorrow',
    legend=False
)

plt.title('Distribucija RainTomorrow')
plt.xlabel('Da li će sutra padati kiša')
plt.ylabel('Broj zapisa')

plt.tight_layout()
plt.show()

### 6.5 Korelacije promenljivih

#### 6.5.1 Pirsonov koeficijent

In [ ]:
plt.figure(figsize=(12, 8))
corr_matrix = data.corr(numeric_only=True)

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', mask=mask)
for i in range(len(corr_matrix)):
    plt.text(i + 0.5, i + 0.5, "1.00", ha="center", va="center", color="black")
plt.title("Matrica korelacija (Pearson)")
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()

Iz prikazane Pearson matrice uočavamo sledeće:

- MaxTemp i Temp3pm pokazuju veoma jaku pozitivnu korelaciju (r = 0.99), što je očekivano jer je Temp3pm skoro uvek najtopliji deo dana, blizu dnevnog maksimuma.
- Pressure9am i Pressure3pm su takođe skoro identični (r = 0.96) - atmosferski pritisak se retko menja drastično u toku jednog dana.
- MinTemp-Temp9am (r = 0.90), MaxTemp-Temp9am (r = 0.89) i Temp9am-Temp3pm (r = 0.86) pokazuju da su sve četiri temperaturske kolone međusobno jako povezane, što ima smisla jer sve mere istu fizičku veličinu u različitim trenucima dana.
- Sunshine pokazuje umerenu do jaku negativnu korelaciju sa Cloud3pm (r = -0.70) i Cloud9am (r = -0.68) - logično, više oblaka znači manje sunčevih sati.
- WindGustSpeed ima samo umerenu korelaciju sa WindSpeed9am/3pm (r ≈ 0.60-0.69) - jak udar vetra ne znači nužno i konstantno jak vetar tokom dana.

MaxTemp-Temp3pm, Pressure9am-Pressure3pm, MinTemp-Temp9am, MaxTemp-Temp9am i Temp9am-Temp3pm sa jakom korelacijom mogu potencijalno da predstavljaju problem multikolinearnosti.
#### 6.5.2 Spirmanov koeficijent
S obzirom da smo ranije utvrdili da je Rainfall ima jaku desnu asimetriju, bilo bi dobro proveriti Spirmanov koeficijent korelacije.

In [ ]:
corr_spearman = data.corr(method='spearman', numeric_only=True)
razlika = (corr_spearman - corr_matrix)

mask_r = np.triu(np.ones_like(razlika, dtype=bool))
razlika_masked = razlika.mask(mask_r)

top_razlike = razlika_masked.unstack().dropna().sort_values(key=lambda x: x.abs(), ascending=False)
top_razlike.head(10)

Poređenje Pearson i Spirman korelacije potvrđuje pretpostavku - najveća odstupanja se dešavaju baš oko kolone Rainfall:

- Rainfall-Evaporation: Pearson r = -0.06, Spirman r = -0.31 (razlika -0.24)
- Rainfall-Temp3pm: Pearson r = -0.08, Spirman r = -0.31
- Rainfall-MaxTemp: Pearson r = -0.08, Spirman r = -0.30
- Rainfall-Humidity9am: Pearson r = 0.22, Spirman r = 0.44 (razlika +0.22)
- Rainfall-Sunshine: Pearson r = -0.23, Spirman r = -0.40

Postoje nelinearne (ali monotone) zavisnosti u skupu podataka, prvenstveno vezane za Rainfall, i za tu kolonu je Spirman korelacija pouzdaniji pokazatelj nego Pearson. Za ostale, približno normalno raspodeljene kolone (temperature, pritisak), Pearson i Spirman se gotovo poklapaju, pa tu razlika nije bitna. Međutim, čak i sa Spirmanovim koeficijentom, vidimo da nije dovoljna korelacija Rainfall-a sa ostalim promenljivama.

#### 6.5.3 VIF faktor

In [ ]:

num = data.select_dtypes("number").dropna()
num_const = sm.add_constant(num)

vif = pd.DataFrame()
vif["kolona"] = num_const.columns
vif["VIF"] = [variance_inflation_factor(num_const.values, i) for i in range(num_const.shape[1])]
vif = vif[vif["kolona"] != "const"].sort_values("VIF", ascending=False)
vif

Rezultati potvrđuju ono što smo već videli kroz Pearson korelaciju: Temp3pm (56.4), MaxTemp (47.2), Temp9am (24.6), Pressure9am (20.1), Pressure3pm (20.1) i MinTemp (11.0) prelaze granicu od 10 - problem multikolinearnosti. Ostale kolone (Humidity, Sunshine, WindGustSpeed, Cloud, Evaporation, WindSpeed, Rainfall) imaju VIF ispod 7, što je prihvatljivo.


Multikolinearnost nam predstavlja problem posebno za linearne modele, gde bismo ukoliko njih koristimo, trebalo da izbacimo jedan po jedan ili da kombinujemo više njih u jedan.

### 6.6 Vremenska (temporalna) analiza

Kolone Year, Month i DayOfYear već smo izdvojili iz Date u sekciji 4. Pogledajmo da li postoji sezonski obrazac u učestalosti kiše, i da li postoji trend kroz godine (2007-2017) - ovo je relevantno i za odabir tačke podele na trening/test skup u odeljku 7.

In [ ]:
mesec_nazivi = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'Maj', 6:'Jun',
                7:'Jul', 8:'Avg', 9:'Sep', 10:'Okt', 11:'Nov', 12:'Dec'}

rain_rate_month = data.groupby('Month')['RainTomorrow'].apply(lambda s: (s == 'Yes').mean() * 100)

plt.figure(figsize=(9, 4))
sns.barplot(
    x=rain_rate_month.index.map(mesec_nazivi),
    y=rain_rate_month.values,
    color=sns.color_palette('Set1')[0]
)
plt.title('Procenat dana sa kišom sutradan, po mesecu')
plt.xlabel('Mesec')
plt.ylabel('Procenat kiše sutra (%)')
plt.tight_layout()
plt.show()

Postoji jasan sezonski obrazac: šansa za kišu sutradan raste sa oko 19% u januaru do oko 26% u julu, tj. tokom južnohemisferske zime (jun-avgust), i ponovo opada ka proleću/leti - razlika od skoro 7 procentnih poena između najsušnijeg i najvlažnijeg meseca. Ovo potvrđuje da je sezonalnost stvaran signal.

In [ ]:
rain_rate_year = data.groupby('Year')['RainTomorrow'].apply(lambda s: (s == 'Yes').mean() * 100)
broj_redova_godina = data.groupby('Year').size()

fig, ax1 = plt.subplots(figsize=(9, 4))
sns.lineplot(x=rain_rate_year.index, y=rain_rate_year.values, marker='o', ax=ax1, color=sns.color_palette('Set1')[0])
ax1.set_xlabel('Godina')
ax1.set_ylabel('Procenat kiše sutra (%)')
ax1.set_title('Procenat dana sa kišom sutradan, po godini')

for x, y, n in zip(rain_rate_year.index, rain_rate_year.values, broj_redova_godina.values):
    ax1.annotate(f'n={n}', (x, y), textcoords='offset points', xytext=(0, 8), fontsize=7, ha='center')

plt.tight_layout()
plt.show()

Skup podataka počinje 1.11.2007. i završava se 25.6.2017, pa su 2007. (svega 61 red, samo novembar-decembar) i 2017. (do juna, 8623 reda) parcijalne godine i njihov procenat kiše nije pouzdano uporediv sa punim godinama. Za pune godine (2008-2016) procenat kiše sutradan varira između otprilike 20% i 24%, bez jasnog dugoročnog uzlaznog ili silaznog trenda.